# 00 - Setup Validation Notebook
Run this first to confirm environment, secrets, and storage paths are ready.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

CONFIG = {
    "secret_scope": "trustpilot",
    "secret_key": "api_key",
    "catalog": "avant_users",
    "schema": "kaley_ubellacker",
    "bronze_table": "trustpilot_reviews_bronze",
    "silver_table": "trustpilot_reviews_silver",
    "gold_table": "trustpilot_sentiment_gold",
    "csv_output": "/Volumes/avant_users/kaley_ubellacker/sentiment_analysis/trustpilot_master_reviews/",
}

In [0]:
print("Step 1/5: Spark session")
print(f"Spark version: {spark.version}")

print("\nStep 2/5: Secret availability")
try:
    token = dbutils.secrets.get(scope=CONFIG["secret_scope"], key=CONFIG["secret_key"])
    masked = token[:4] + "..." if token else "<empty>"
    print(f"✅ Secret resolved: {masked}")
except Exception as e:
    print("❌ Unable to read secret. Create scope/key before ingestion.")
    raise

print("\nStep 3/5: Schema bootstrap")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CONFIG['catalog']}.{CONFIG['schema']}")
print("✅ Schema ready")

print("\nStep 4/5: Table existence checks")
for t in [CONFIG["bronze_table"], CONFIG["silver_table"], CONFIG["gold_table"]]:
    full = f"{CONFIG['catalog']}.{CONFIG['schema']}.{t}"
    exists = spark.catalog.tableExists(full)
    print(f"- {full}: {'FOUND' if exists else 'NOT FOUND (expected before first run)'}")

print("\nStep 5/5: Storage path write-read smoke test")
smoke_path = CONFIG["csv_output"] + "/_smoke_test"
(
    spark.createDataFrame([("ok",)], ["status"])
    .withColumn("ts", F.current_timestamp())
    .coalesce(1)
    .write.mode("overwrite").option("header", True).csv(smoke_path)
)
read_back = spark.read.option("header", True).csv(smoke_path)
print(f"✅ Storage smoke row count: {read_back.count()}")

print("\nValidation complete. Next run notebook: 01_trustpilot_ingestion.py")